In [ ]:
!ls

fuel_delay_project.zip	sample_data


In [ ]:
!unzip fuel_delay_project.zip -d fuel_delay_project

Archive:  fuel_delay_project.zip
  inflating: fuel_delay_project/main.py  
  inflating: fuel_delay_project/README.md  
  inflating: fuel_delay_project/pareto.py  
   creating: fuel_delay_project/outputs/
  inflating: fuel_delay_project/outputs/speed_cost_curve_Fishing_Trawler.png  
  inflating: fuel_delay_project/outputs/speed_cost_curve_Oil_Service_Boat.png  
  inflating: fuel_delay_project/outputs/pareto_frontier_Tanker_Ship.png  
  inflating: fuel_delay_project/outputs/model_comparison.png  
  inflating: fuel_delay_project/outputs/feature_importance.png  
  inflating: fuel_delay_project/outputs/pareto_frontier_Oil_Service_Boat.png  
  inflating: fuel_delay_project/outputs/speed_cost_curve_Tanker_Ship.png  
  inflating: fuel_delay_project/outputs/pareto_frontier_Fishing_Trawler.png  
  inflating: fuel_delay_project/outputs/pareto_frontier_Surfer_Boat.png  
  inflating: fuel_delay_project/outputs/speed_cost_curve_Surfer_Boat.png  
   creating: fuel_delay_project/data/
  inflating: fue

In [ ]:
%cd fuel_delay_project

/content/fuel_delay_project


In [ ]:
!ls

data	      feature_engineering.py  models.py  pareto.py
data_prep.py  main.py		      outputs	 README.md


In [ ]:
!pip install tensorflow

In [ ]:
!python3 main.py

DATA SOURCE: SYNTHETIC placeholder data
  -> Drop the real CSV at data/ship_fuel_efficiency.csv and re-run
     to replace all numbers below with real-data results.

[1/4] Training Polynomial Regression...
  {'model': 'Polynomial Regression (deg=2)', 'MAE': 1878.5696277748398, 'R2': 0.8142350982755244}

[2/4] Training Random Forest...
  {'model': 'Random Forest', 'MAE': 1364.124301813089, 'R2': 0.884345083242469}
  Saved: /content/fuel_delay_project/outputs/feature_importance.png

[3/4] Checking LSTM feasibility...
  Feasible: True (voyages/ship -- min:36, median:36.0, max:36)
2026-07-18 16:06:48.045664: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-18 16:06:48.120066: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild Tens

In [ ]:
!ls /content/fuel_delay_project/data/

ship_fuel_efficiency.csv


In [ ]:
%cd /content/fuel_delay_project
!python3 main.py

/content/fuel_delay_project
Traceback (most recent call last):
  File "/content/fuel_delay_project/main.py", line 168, in <module>
    run()
  File "/content/fuel_delay_project/main.py", line 111, in run
    raw, is_real = load_data()
                   ^^^^^^^^^^^
  File "/content/fuel_delay_project/data_prep.py", line 133, in load_data
    df = pd.read_csv(DATA_PATH, parse_dates=["date"])
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1026, in read_csv
    return _read(filepath_or_buffer, kwds)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 620, in _read
    parser = TextFileReader(filepath_or_buffer, **kwds)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pandas/io/parsers/readers.py", line 1620, in __init__
    self._engine = self._make_engine(f, self.engine)

In [ ]:
import pandas as pd
df = pd.read_csv('/content/fuel_delay_project/data/ship_fuel_efficiency.csv')
print(df.columns.tolist())
print(df.head())

['ship_id', 'ship_type', 'route_id', 'month', 'distance', 'fuel_type', 'fuel_consumption', 'CO2_emissions', 'weather_conditions', 'engine_efficiency']
  ship_id         ship_type             route_id     month  distance  \
0   NG001  Oil Service Boat          Warri-Bonny   January    132.26   
1   NG001  Oil Service Boat  Port Harcourt-Lagos  February    128.52   
2   NG001  Oil Service Boat  Port Harcourt-Lagos     March     67.30   
3   NG001  Oil Service Boat  Port Harcourt-Lagos     April     71.68   
4   NG001  Oil Service Boat          Lagos-Apapa       May    134.32   

  fuel_type  fuel_consumption  CO2_emissions weather_conditions  \
0       HFO           3779.77       10625.76             Stormy   
1       HFO           4461.44       12779.73           Moderate   
2       HFO           1867.73        5353.01               Calm   
3    Diesel           2393.51        6506.52             Stormy   
4       HFO           4267.19       11617.03               Calm   

   engine_eff

In [ ]:
%%writefile /content/fuel_delay_project/data_prep.py
"""
data_prep.py
------------
Loads the ship fuel dataset. If the real Kaggle CSV
(ship_fuel_efficiency.csv from "Ship Fuel Consumption & CO2 Emissions
Analysis") is present in ./data/, it's used directly. Otherwise a
synthetic dataset is generated that matches the REAL dataset's documented
schema and value ranges, so the rest of the pipeline (feature engineering,
models, Pareto frontier) can be built and tested before the real file
is dropped in.
"""

import os
import numpy as np
import pandas as pd

DATA_PATH = os.path.join(os.path.dirname(__file__), "data", "ship_fuel_efficiency.csv")

SHIP_TYPES = ["Oil Service Boat", "Fishing Trawler", "Surfer Boat", "Tanker Ship"]
FUEL_TYPES = ["Diesel", "HFO"]
WEATHER = ["Calm", "Moderate", "Stormy"]

NOMINAL_SPEED_KNOTS = {
    "Oil Service Boat": 10.0,
    "Fishing Trawler": 8.0,
    "Surfer Boat": 20.0,
    "Tanker Ship": 14.0,
}


def _generate_synthetic(n_ships=40, voyages_per_ship=36, seed=42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    rows = []
    start_date = pd.Timestamp("2023-01-01")

    for ship_idx in range(n_ships):
        ship_id = f"SHIP_{ship_idx:03d}"
        ship_type = rng.choice(SHIP_TYPES)
        nominal_speed = NOMINAL_SPEED_KNOTS[ship_type]
        fuel_type = rng.choice(FUEL_TYPES, p=[0.55, 0.45])
        fouling_drift = 0.0
        last_drydock = 0
        voyage_date = start_date + pd.Timedelta(days=int(rng.integers(0, 60)))

        for v in range(voyages_per_ship):
            distance = float(np.clip(rng.normal(220, 110), 20, 499))
            weather = rng.choice(WEATHER, p=[0.5, 0.35, 0.15])
            weather_drag = {"Calm": 1.00, "Moderate": 1.08, "Stormy": 1.22}[weather]
            engine_efficiency = float(np.clip(rng.normal(85, 6), 70, 95))
            true_speed = max(3.0, rng.normal(nominal_speed, nominal_speed * 0.12))
            base_k = {"Oil Service Boat": 0.23, "Fishing Trawler": 0.28,
                      "Surfer Boat": 0.035, "Tanker Ship": 0.35}[ship_type]
            efficiency_penalty = (100 - engine_efficiency) / 100 * 0.6 + 1.0
            fouling_drift += rng.normal(0.004, 0.001)
            if v - last_drydock > 20 and rng.random() < 0.15:
                fouling_drift = 0.0
                last_drydock = v
            fuel_consumption = (
                base_k * (true_speed ** 2) * distance
                * weather_drag * efficiency_penalty
                * (1 + fouling_drift) * rng.normal(1.0, 0.04)
            )
            fuel_consumption = float(np.clip(fuel_consumption, 237.88, 24648.52))
            co2_factor = 2.68 if fuel_type == "Diesel" else 3.11
            co2_emissions = fuel_consumption * co2_factor * rng.normal(1.0, 0.03)
            rows.append({
                "date": voyage_date, "ship_id": ship_id, "ship_type": ship_type,
                "route_id": f"R{rng.integers(1, 12)}", "distance": round(distance, 2),
                "fuel_type": fuel_type, "fuel_consumption": round(fuel_consumption, 2),
                "CO2_emissions": round(co2_emissions, 2), "weather_conditions": weather,
                "engine_efficiency": round(engine_efficiency, 2),
            })
            voyage_date = voyage_date + pd.Timedelta(days=int(rng.integers(3, 12)))

    return pd.DataFrame(rows)


MONTH_ORDER = {
    "January": 1, "February": 2, "March": 3, "April": 4, "May": 5, "June": 6,
    "July": 7, "August": 8, "September": 9, "October": 10, "November": 11, "December": 12,
}


def load_data() -> tuple[pd.DataFrame, bool]:
    if os.path.exists(DATA_PATH):
        df = pd.read_csv(DATA_PATH)
        if "date" not in df.columns and "month" in df.columns:
            month_num = df["month"].map(MONTH_ORDER)
            df["date"] = pd.to_datetime({"year": 2023, "month": month_num, "day": 1})
        elif "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"])
        return df, True
    else:
        return _generate_synthetic(), False


if __name__ == "__main__":
    df, is_real = load_data()
    print(f"Loaded {'REAL' if is_real else 'SYNTHETIC (placeholder)'} data: {df.shape}")
    print(df.head())

Overwriting /content/fuel_delay_project/data_prep.py


In [ ]:
!python3 main.py

DATA SOURCE: REAL Kaggle dataset

[1/4] Training Polynomial Regression...
  {'model': 'Polynomial Regression (deg=2)', 'MAE': 737.7261253073963, 'R2': 0.9529976813514788}

[2/4] Training Random Forest...
  {'model': 'Random Forest', 'MAE': 674.3209425027043, 'R2': 0.9484041059338971}
  Saved: /content/fuel_delay_project/outputs/feature_importance.png

[3/4] Checking LSTM feasibility...
  Feasible: True (voyages/ship -- min:12, median:12.0, max:12)
2026-07-18 16:44:50.857193: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-18 16:44:50.944867: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-18 16:44:54.226131: E external/local_xla/xla/stream_executor/cuda/cuda_platform.

In [ ]:
%%writefile /content/fuel_delay_project/pareto.py
"""
pareto.py
---------
Sweeps candidate speeds for a given ship type + distance + weather
scenario, computes fuel, delay, and cost at each speed, finds the
cost-minimizing speed, and compares it against nominal service speed
to quantify savings.
"""

import numpy as np
import pandas as pd

from data_prep import NOMINAL_SPEED_KNOTS
from feature_engineering import FUEL_PRICE_USD_PER_LITRE, DEMURRAGE_USD_PER_HOUR


def _calibrate_k(df, ship_type, admiralty_exponent=2.0):
    subset = df[df["ship_type"] == ship_type]
    nominal_speed = NOMINAL_SPEED_KNOTS[ship_type]
    med_fuel = subset["fuel_consumption"].median()
    med_dist = subset["distance"].median()
    return med_fuel / (nominal_speed ** admiralty_exponent * med_dist)


def speed_cost_curve(df, ship_type, distance, fuel_type="HFO",
                      weather="Moderate", speed_range=None, admiralty_exponent=2.0):
    if speed_range is None:
        nominal = NOMINAL_SPEED_KNOTS[ship_type]
        speed_range = np.linspace(nominal * 0.25, nominal * 1.5, 80)

    weather_drag = {"Calm": 1.00, "Moderate": 1.08, "Stormy": 1.22}[weather]
    k = _calibrate_k(df, ship_type, admiralty_exponent)
    nominal_speed = NOMINAL_SPEED_KNOTS[ship_type]
    fuel_price = FUEL_PRICE_USD_PER_LITRE[fuel_type]
    demurrage_rate = DEMURRAGE_USD_PER_HOUR[ship_type]

    rows = []
    for speed in speed_range:
        fuel = k * (speed ** admiralty_exponent) * distance * weather_drag
        actual_time = distance / speed
        expected_time = distance / nominal_speed
        delay = max(0.0, actual_time - expected_time)
        bunker_cost = fuel * fuel_price
        demurrage_cost = delay * demurrage_rate
        total_cost = bunker_cost + demurrage_cost
        rows.append({
            "speed_knots": speed, "fuel_consumption_l": fuel, "delay_hours": delay,
            "bunker_cost_usd": bunker_cost, "demurrage_cost_usd": demurrage_cost,
            "total_cost_usd": total_cost,
        })

    return pd.DataFrame(rows)


def recommend_speed(df, ship_type, distance, fuel_type="HFO", weather="Moderate"):
    curve = speed_cost_curve(df, ship_type, distance, fuel_type, weather)
    best = curve.loc[curve["total_cost_usd"].idxmin()]

    nominal_speed = NOMINAL_SPEED_KNOTS[ship_type]
    nominal_row = curve.iloc[(curve["speed_knots"] - nominal_speed).abs().idxmin()]

    savings_usd = nominal_row["total_cost_usd"] - best["total_cost_usd"]
    savings_pct = 100 * savings_usd / nominal_row["total_cost_usd"]

    comparison = {
        "nominal_speed_knots": nominal_row["speed_knots"],
        "nominal_total_cost_usd": nominal_row["total_cost_usd"],
        "optimal_speed_knots": best["speed_knots"],
        "optimal_total_cost_usd": best["total_cost_usd"],
        "savings_usd": savings_usd,
        "savings_pct": savings_pct,
    }
    return best, curve, comparison


if __name__ == "__main__":
    from data_prep import load_data
    from feature_engineering import engineer_all

    raw, is_real = load_data()
    df = engineer_all(raw)

    for ship_type in NOMINAL_SPEED_KNOTS:
        best, curve, comparison = recommend_speed(df, ship_type, distance=250)
        print(f"\n{ship_type}")
        print(f"  Nominal ({comparison['nominal_speed_knots']:.1f} kn): ${comparison['nominal_total_cost_usd']:.0f}")
        print(f"  Optimal ({comparison['optimal_speed_knots']:.1f} kn): ${comparison['optimal_total_cost_usd']:.0f}")
        print(f"  SAVINGS: ${comparison['savings_usd']:.0f} ({comparison['savings_pct']:.1f}%)")

Overwriting /content/fuel_delay_project/pareto.py


In [ ]:
with open('/content/fuel_delay_project/main.py', 'r') as f:
    content = f.read()

old = '''    print("\\n[4/4] Building speed-cost tradeoff curves and Pareto frontiers...")
    for ship_type in NOMINAL_SPEED_KNOTS:
        best, curve = recommend_speed(df, ship_type, distance=250)
        p1 = plot_speed_cost_curve(df, ship_type, distance=250)
        p2 = plot_pareto_frontier(df, ship_type, distance=250)
        print(f"\\n  {ship_type} (250nm voyage):")
        print(f"    Recommended speed: {best['speed_knots']:.1f} kn "
              f"(nominal: {NOMINAL_SPEED_KNOTS[ship_type]:.1f} kn)")
        print(f"    Total cost at optimum: ${best['total_cost_usd']:.0f} "
              f"(bunker ${best['bunker_cost_usd']:.0f} + demurrage ${best['demurrage_cost_usd']:.0f})")
        print(f"    Saved: {p1}")
        print(f"    Saved: {p2}")'''

new = '''    print("\\n[4/4] Building speed-cost tradeoff curves and Pareto frontiers...")
    for ship_type in NOMINAL_SPEED_KNOTS:
        best, curve, comparison = recommend_speed(df, ship_type, distance=250)
        p1 = plot_speed_cost_curve(df, ship_type, distance=250)
        p2 = plot_pareto_frontier(df, ship_type, distance=250)
        print(f"\\n  {ship_type} (250nm voyage):")
        print(f"    Nominal speed {comparison['nominal_speed_knots']:.1f} kn -> ${comparison['nominal_total_cost_usd']:.0f}")
        print(f"    Optimal speed {comparison['optimal_speed_knots']:.1f} kn -> ${comparison['optimal_total_cost_usd']:.0f}")
        print(f"    SAVINGS: ${comparison['savings_usd']:.0f} ({comparison['savings_pct']:.1f}%)")
        print(f"    Saved: {p1}")
        print(f"    Saved: {p2}")'''

content = content.replace(old, new)
with open('/content/fuel_delay_project/main.py', 'w') as f:
    f.write(content)
print("patched")

patched


In [ ]:
!python3 main.py

DATA SOURCE: REAL Kaggle dataset

[1/4] Training Polynomial Regression...
  {'model': 'Polynomial Regression (deg=2)', 'MAE': 737.7261253073963, 'R2': 0.9529976813514788}

[2/4] Training Random Forest...
  {'model': 'Random Forest', 'MAE': 674.3209425027043, 'R2': 0.9484041059338971}
  Saved: /content/fuel_delay_project/outputs/feature_importance.png

[3/4] Checking LSTM feasibility...
  Feasible: True (voyages/ship -- min:12, median:12.0, max:12)
2026-07-18 16:53:43.978539: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-07-18 16:53:44.064809: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-07-18 16:53:47.257130: E external/local_xla/xla/stream_executor/cuda/cuda_platform.

In [ ]:
%%writefile /content/fuel_delay_project/demo.py
"""
demo.py
-------
Live demo for viva: pick a ship type, distance, weather, and fuel type,
and instantly see the predicted fuel consumption, recommended speed, and
cost savings vs sailing at nominal speed.
"""

from data_prep import load_data, NOMINAL_SPEED_KNOTS, FUEL_TYPES, WEATHER
from feature_engineering import engineer_all
from pareto import recommend_speed, speed_cost_curve
import matplotlib.pyplot as plt


def demo_run(ship_type="Tanker Ship", distance=250, weather="Moderate", fuel_type="HFO",
             df=None, show_plot=True):
    if df is None:
        raw, is_real = load_data()
        df = engineer_all(raw)

    best, curve, comparison = recommend_speed(df, ship_type, distance, fuel_type, weather)

    print("=" * 60)
    print(f"SCENARIO: {ship_type} | {distance:.0f} nm | {weather} weather | {fuel_type}")
    print("=" * 60)
    print(f"Nominal service speed : {comparison['nominal_speed_knots']:.1f} kn")
    print(f"  -> predicted cost    : ${comparison['nominal_total_cost_usd']:,.0f}")
    print()
    print(f"RECOMMENDED speed     : {comparison['optimal_speed_knots']:.1f} kn")
    print(f"  -> predicted cost    : ${comparison['optimal_total_cost_usd']:,.0f}")
    print()
    print(f"SAVINGS                : ${comparison['savings_usd']:,.0f} ({comparison['savings_pct']:.1f}%)")
    print("=" * 60)

    if show_plot:
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(curve["speed_knots"], curve["bunker_cost_usd"], label="Bunker (fuel) cost", color="#2563eb")
        ax.plot(curve["speed_knots"], curve["demurrage_cost_usd"], label="Demurrage (delay) cost", color="#dc2626")
        ax.plot(curve["speed_knots"], curve["total_cost_usd"], label="Total voyage cost", color="#111827", linewidth=2.5)
        ax.axvline(comparison["optimal_speed_knots"], color="#16a34a", linestyle="--",
                   label=f"Recommended = {comparison['optimal_speed_knots']:.1f} kn")
        ax.axvline(comparison["nominal_speed_knots"], color="#9ca3af", linestyle=":",
                   label=f"Nominal = {comparison['nominal_speed_knots']:.1f} kn")
        ax.set_xlabel("Speed (knots)")
        ax.set_ylabel("Cost (USD)")
        ax.set_title(f"{ship_type} | {distance:.0f}nm | {weather}")
        ax.legend()
        plt.tight_layout()
        plt.show()

    return comparison


def demo_widget():
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    raw, is_real = load_data()
    df = engineer_all(raw)
    print(f"Data source: {'REAL Kaggle dataset' if is_real else 'SYNTHETIC placeholder'}")

    ship_dropdown = widgets.Dropdown(options=list(NOMINAL_SPEED_KNOTS.keys()),
                                      value="Tanker Ship", description="Ship type:")
    distance_slider = widgets.IntSlider(value=250, min=20, max=500, step=10, description="Distance (nm):")
    weather_dropdown = widgets.Dropdown(options=WEATHER, value="Moderate", description="Weather:")
    fuel_dropdown = widgets.Dropdown(options=FUEL_TYPES, value="HFO", description="Fuel type:")
    output = widgets.Output()

    def on_change(change=None):
        with output:
            clear_output(wait=True)
            demo_run(ship_dropdown.value, distance_slider.value,
                     weather_dropdown.value, fuel_dropdown.value, df=df)

    for w in [ship_dropdown, distance_slider, weather_dropdown, fuel_dropdown]:
        w.observe(on_change, names="value")

    display(ship_dropdown, distance_slider, weather_dropdown, fuel_dropdown, output)
    on_change()

Writing /content/fuel_delay_project/demo.py


In [ ]:
%cd /content/fuel_delay_project
from demo import demo_widget
demo_widget()

/content/fuel_delay_project
Data source: REAL Kaggle dataset


Dropdown(description='Ship type:', index=3, options=('Oil Service Boat', 'Fishing Trawler', 'Surfer Boat', 'Ta…

IntSlider(value=250, description='Distance (nm):', max=500, min=20, step=10)

Dropdown(description='Weather:', index=1, options=('Calm', 'Moderate', 'Stormy'), value='Moderate')

Dropdown(description='Fuel type:', index=1, options=('Diesel', 'HFO'), value='HFO')

Output()